## HALI v4 — HPV Awareness & Learning Initiative
### AutoGen AgentChat | Week 5 Community Contribution

A two-agent review team for Community Health Volunteers in Kenya.

**Pattern:** RoundRobinGroupChat with TextMentionTermination
- `MythBuster` — responds to parent objections with evidence
- `Reviewer` — checks accuracy, cultural fit, and conciseness; approves or requests a rewrite

The team loops until the Reviewer says APPROVED — same evaluator-rerun 
pattern from Week 1, now expressed natively in AutoGen.

**Context:** Cervical cancer kills 3,400 Kenyan women every year.
CHVs need responses they can use immediately in the field.


In [2]:
# Imports and setup
import os
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv(override=True)
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")


In [3]:
# Define the two agents
myth_buster = AssistantAgent(
    name="Mythbuster",
    model_client=model_client,
    system_message="""You are HALI, supporting Community Health Volunteers (CHVs) in Kenya 
with HPV vaccine outreach. When given a question or objection from a parent, 
give a warm, evidence-based response the CHV can use immediately in the field.
Key facts:
- The vaccine does NOT cause infertility — this is the most common myth
- One dose is enough for girls 9-14, it is free at public facilities
- It prevents cervical cancer which kills 3,400 Kenyan women every year
- Islamic and Christian leaders in Kenya support vaccination
Keep your response under 150 words.""",
)

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=model_client,
    system_message="""You are a quality reviewer for HALI, an HPV vaccine support tool for Kenya.
Review the MythBuster's response and check:
1. Is it factually accurate about the HPV vaccine?
2. Is it culturally appropriate for Kenya?
3. Is it short enough for a CHV to use in the field?
If all checks pass, respond with: APPROVED
If not, explain what needs to change so MythBuster can try again.""",
)

In [5]:
# Build the team
termination = TextMentionTermination("APPROVED")

team = RoundRobinGroupChat(
    participants=[myth_buster, reviewer],
    termination_condition=termination,
    max_turns=6,
)


In [6]:
# Test with the most common myth
import asyncio

async def run():
    await Console(
        team.run_stream(
            task="A mother in Kisumu says: 'I won't let my daughter get the vaccine — I heard it causes infertility.' What do I tell her?"
        )
    )

await run()

---------- TextMessage (user) ----------
A mother in Kisumu says: 'I won't let my daughter get the vaccine — I heard it causes infertility.' What do I tell her?
---------- TextMessage (Mythbuster) ----------
You can reassure her by saying: "I understand your concern, but I want to share that many studies have shown the HPV vaccine does not cause infertility. In fact, it is a safe and effective way to protect your daughter from cervical cancer, which unfortunately affects many women in our country. The vaccine is free at public health facilities and only one dose is needed for girls aged 9-14. Additionally, both Islamic and Christian leaders in Kenya encourage vaccination for its health benefits. Let’s work together to keep our daughters safe and healthy."
---------- TextMessage (Reviewer) ----------
The response has several strengths, but there are a few areas for improvement:

1. **Factual Accuracy**: The response correctly emphasizes that the HPV vaccine does not cause infertility an